<font size=10>**NETWORK**</font> <a class="anchor" id='title'></a> 

**Bachelor's in Data Science - NOVA IMS (25/26)**

<font color='#BFD72' size=5>**RESEARCH QUESTION**: </font><font size=5>*Which companies have a dominant position within specific municipalities?*</font> 

**Data**: 
- [*Portal BASE*](https://www.base.gov.pt/Base4/pt/pesquisa/?type=contratos&texto=&adjudicante=&adjudicataria=&tipo=2&tipocontrato=0&cpv=&aqinfo=&desdeprazoexecucao=&ateprazoexecucao=&sel_price=price_11&desdeprecocontrato=&ateprecocontrato=&desdeprecoefectivo=&ateprecoefectivo=&sel_date=date_11&desdedatacontrato=2023-01-01&atedatacontrato=2026-03-31&desdedatapublicacao=&atedatapublicacao=&desdedatafecho=&atedatafecho=&pais=0&distrito=0&concelho=0)

- [*Treated Datasets*](https://dados.gov.pt/pt/datasets/contratos-publicos-portal-base-impic-contratos-de-2012-a-2026/#/resources)

**Group B**
- Beatriz Marques 20231605
- Maria Inês Santos 20231630
- Luís Soeiro 20211536
- Rodrigo Silva 20231602

<font color='#BFD72' size=6>**TABLE OF CONTENTS**</font> <a class="anchor" id='toc'></a>  
- [1. Imports](#1)  
- [2. Data Integration](#2)  
- [3. First Network](#3)

# <font color='#BFD72F' size=6>**1. Imports**</font> <a class="anchor" id="1"></a>

[Back to TOC](#toc)

In [1]:
import warnings
%load_ext autoreload
%autoreload 2

warnings.filterwarnings('ignore')

Failed to read module file 'c:\Users\Rafael\AppData\Local\Programs\Python\Python312\Lib\pydoc_data\topics.py' for module 'pydoc_data.topics': UnicodeDecodeError
Traceback (most recent call last):
  File "C:\Users\Rafael\AppData\Roaming\Python\Python312\site-packages\IPython\core\extensions.py", line 62, in load_extension
    return self._load_extension(module_str)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\Rafael\AppData\Roaming\Python\Python312\site-packages\IPython\core\extensions.py", line 77, in _load_extension
    mod = import_module(module_str)
          ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\Rafael\AppData\Local\Programs\Python\Python312\Lib\importlib\__init__.py", line 90, in import_module
    return _bootstrap._gcd_import(name[level:], package, level)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "<frozen importlib._bootstrap>", line 1387, in _gcd_import
  File "<frozen importlib._bootstrap>", line 1360, in _find_and_load
  File "

In [2]:
import sys
import os

# Get the absolute path of the source_code folder
source_code_path = os.path.abspath('../source')

# Add the source_code folder to sys.path
if source_code_path not in sys.path:
    sys.path.append(source_code_path)

In [3]:
import subprocess, sys, importlib
import pandas as pd
import plotly.express as px
import re
import networkx as nx
import matplotlib.pyplot as plt
import numpy as np
import plotly.graph_objects as go

try:
    import openpyxl
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "openpyxl"])
    importlib.invalidate_caches()
    import openpyxl
    
import os
import shutil

# <font color='#BFD72F' size=6>**2. Data Integration**</font> <a class="anchor" id="2"></a>
  
[Back to TOC](#toc)

In [4]:
data = pd.read_csv('../data/preprocessed_data.csv')
# data.head()
data.info()

<class 'pandas.DataFrame'>
RangeIndex: 16989 entries, 0 to 16988
Data columns (total 25 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   idcontrato                   16989 non-null  int64  
 1   tipoContrato                 16989 non-null  str    
 2   tipoFimContrato              16972 non-null  str    
 3   CPV                          16989 non-null  str    
 4   adjudicante                  16989 non-null  str    
 5   adjudicatarios               16989 non-null  str    
 6   concorrentes                 14707 non-null  str    
 7   precoBaseProcedimento        16989 non-null  float64
 8   precoContratual              16989 non-null  float64
 9   PrecoTotalEfetivo            16989 non-null  float64
 10  LocalExecucao                16989 non-null  str    
 11  dataDecisaoAdjudicacao       16989 non-null  str    
 12  dataCelebracaoContrato       16989 non-null  str    
 13  dataPublicacao             

In [5]:
data["dataPublicacao"] = pd.to_datetime(data["dataPublicacao"], errors='coerce')
data["dataCelebracaoContrato"] = pd.to_datetime(data["dataCelebracaoContrato"], errors='coerce')
data["dataDecisaoAdjudicacao"] = pd.to_datetime(data["dataDecisaoAdjudicacao"], errors='coerce')
data["dataFechoContrato"] = pd.to_datetime(data["dataFechoContrato"], errors='coerce')

# <font color='#BFD72F' size=6>**3. The Network**</font> <a class="anchor" id="3"></a>
  
[Back to TOC](#toc)

## <font size=5>**3.2 Public entities / Companies network**</font> <a class="anchor" id="3.1"></a>
  
[Back to TOC](#toc)

In [6]:
################
### BEA CODE ###
################

# # If your data is not a DataFrame, fix it
# if isinstance(data, dict):
#     data = pd.DataFrame(data)

# # Create directed graph
# G = nx.DiGraph()

# # Build graph
# for _, row in data.iterrows():
#     adjudicante = row['adjudicante']
#     adjudicatarios = row['adjudicatarios']
#     preco = row['precoContratual']
#     concorrentes = row.get('concorrentes', 0)

#     # Skip missing
#     if pd.isna(adjudicante) or pd.isna(adjudicatarios):
#         continue
#     if pd.isna(preco) or preco <= 0:
#         continue

#     log_price = np.log(preco)

#     # Handle concorrentes safely
#     if pd.isna(concorrentes):
#         concorrentes = 0

#     # Add/update nodes
#     if adjudicante not in G:
#         G.add_node(adjudicante, node_type='adjudicante')
#     else:
#         if G.nodes[adjudicante].get('node_type') == 'adjudicatario':
#             G.nodes[adjudicante]['node_type'] = 'both'

#     if adjudicatarios not in G:
#         G.add_node(adjudicatarios, node_type='adjudicatario')
#     else:
#         if G.nodes[adjudicatarios].get('node_type') == 'adjudicante':
#             G.nodes[adjudicatarios]['node_type'] = 'both'

#     # Add/update edge
#     if G.has_edge(adjudicante, adjudicatarios):
#         edge = G[adjudicante][adjudicatarios]
#         edge['weight'] += log_price
#         edge['contracts'] += 1
#         edge_conc = pd.to_numeric(edge.get('nr_concorrentes', 0), errors='coerce')
#         row_conc = pd.to_numeric(concorrentes, errors='coerce')
#         edge['nr_concorrentes'] = int(0 if pd.isna(edge_conc) else edge_conc) + int(0 if pd.isna(row_conc) else row_conc)
#     else:
#         G.add_edge(
#             adjudicante,
#             adjudicatarios,
#             weight=log_price,
#             contracts=1,
#             nr_concorrentes=concorrentes,
#             city=row.get('city', None)
#         )

In [7]:
def compute_weight(prices, mode="log_sum"):

    prices = np.array(prices)

    if mode == "sum_price":
        return prices.sum()

    elif mode == "avg_price":
        return prices.mean()

    elif mode == "log_sum":
        return np.log(prices).sum()

    elif mode == "log_mean":
        return np.log(prices).mean()

    else:
        raise ValueError(
            f"Unknown mode: {mode}. "
            f"Valid modes: sum_price, avg_price, log_sum, log_mean"
        )

In [8]:
def _ensure_dataframe(data):
    if isinstance(data, pd.DataFrame):
        return data.copy()

    if isinstance(data, dict):
        # CASE 1: dict of lists (valid dataframe)
        try:
            df = pd.DataFrame(data)
            return df
        except Exception:
            pass

        # CASE 2: dict of scalars → wrap into list
        return pd.DataFrame([data])

    raise TypeError(f"Unsupported input type: {type(data)}")

In [9]:
from collections import Counter

def most_common(x):
    if isinstance(x, list) and len(x) > 0:
        return Counter(x).most_common(1)[0][0]
    return "Outros / Não classificado"

In [10]:
def build_contract_network(data, weight_mode="log_sum") -> nx.DiGraph:
    
    df = _ensure_dataframe(data)

    # --- CLEAN COLUMNS ---
    df = df.rename(columns={
        'precoContratual': 'price',
        'adjudicante_clean': 'source',
        'adjudicatarios_clean': 'target'
    })

    # --- BASIC CLEANING ---
    df = df.dropna(subset=['source', 'target', 'price'])
    df = df[df['price'] > 0]

    # --- CONCORRENTES ---
    if 'nr_concorrentes' in df.columns:
        df['concorrentes'] = df['nr_concorrentes']
    else:
        df['concorrentes'] = pd.to_numeric(df.get('concorrentes', 0), errors='coerce').fillna(0)

    # --- EDGE AGGREGATION ---
    edge_df = (
        df.groupby(['source', 'target'], as_index=False)
        .agg(
            total_price=('price', 'sum'),
            nr_concorrentes=('concorrentes', 'sum'),
            contracts=('price', 'count'),
            price_series=('price', list),
            idcontrato=('idcontrato', list),
            tipoContrato=('tipoContrato', list),
            tipoFimContrato=('tipoFimContrato', list),
            CPV=('CPV', list),
            precoBaseProcedimento=('precoBaseProcedimento', list),
            precoContratual=('price', list),
            PrecoTotalEfetivo=('PrecoTotalEfetivo', list),
            dataDecisaoAdjudicacao=('dataDecisaoAdjudicacao', list),
            dataCelebracaoContrato=('dataCelebracaoContrato', list),
            dataPublicacao=('dataPublicacao', list),
            dataFechoContrato=('dataFechoContrato', list),
            nr_concorrentes_list=('nr_concorrentes', list),
            contribuinte_adjudicante=('contribuinte_adjudicante', list),
            contribuinte_adjudicatarios=('contribuinte_adjudicatarios', list),
            city=('city', lambda x: Counter(x).most_common(1)[0][0]),
            cpv_prefix=('cpv_prefix', list),
            agg_cpv=('agg_cpv', lambda x: Counter(x).most_common(1)[0][0])
        )
    )

    # --- WEIGHT STRATEGY ---
    edge_df['weight'] = edge_df['price_series'].apply(
        lambda x: compute_weight(x, mode=weight_mode)
    )

    edge_df = edge_df.drop(columns=['price_series'])

    # --- BUILD GRAPH ---
    G = nx.DiGraph()

    for row in edge_df.itertuples(index=False):
        G.add_edge(
            row.source,
            row.target,
            weight=row.weight,
            total_price=row.total_price,
            nr_concorrentes=row.nr_concorrentes,
            contracts=row.contracts,
            weight_mode=weight_mode,
            idcontrato=row.idcontrato,
            tipoContrato=row.tipoContrato,
            tipoFimContrato=row.tipoFimContrato,
            CPV=row.CPV,
            precoBaseProcedimento=row.precoBaseProcedimento,
            precoContratual=row.precoContratual,
            PrecoTotalEfetivo=row.PrecoTotalEfetivo,
            dataDecisaoAdjudicacao=row.dataDecisaoAdjudicacao,
            dataCelebracaoContrato=row.dataCelebracaoContrato,
            dataPublicacao=row.dataPublicacao,
            dataFechoContrato=row.dataFechoContrato,
            nr_concorrentes_list=row.nr_concorrentes_list,
            contribuinte_adjudicante=row.contribuinte_adjudicante,
            contribuinte_adjudicatarios=row.contribuinte_adjudicatarios,
            city=row.city,
            cpv_prefix=row.cpv_prefix,
            agg_cpv=row.agg_cpv
        )

    # --- NODE TYPES ---
    sources = set(edge_df['source'])
    targets = set(edge_df['target'])

    for node in G.nodes():
        if node in sources and node in targets:
            G.nodes[node]['node_type'] = 'both'
        elif node in sources:
            G.nodes[node]['node_type'] = 'adjudicante'
        else:
            G.nodes[node]['node_type'] = 'adjudicatario'

    return G

# -------------------------------
# VISUAL ATTRIBUTES (UNIFIED)
# -------------------------------
def add_visual_attributes(
    G,
    thickness_mode="linear",
    size_mode="linear",
    scale_min=1,
    scale_max=5,
    suffix=""
):
    conc = np.array([d['nr_concorrentes'] for _, _, d in G.edges(data=True)])
    weight = np.array([d['weight'] for _, _, d in G.edges(data=True)])

    def normalize(x):
        if len(x) == 0:
            return x

        range_ = np.ptp(x)  # max - min safely

        if range_ == 0:
            return np.zeros_like(x)

        return (x - np.min(x)) / (range_ + 1e-9)

    # --- TRANSFORM ---
    if thickness_mode == "log":
        conc = np.log1p(conc)

    if size_mode == "log":
        weight = np.log1p(weight)

    conc_norm = normalize(conc)
    weight_norm = normalize(weight)

    def scale(x):
        return scale_min + (scale_max - scale_min) * x

    # --- ASSIGN ---
    for i, (u, v, d) in enumerate(G.edges(data=True)):
        d[f'edge_thickness{suffix}'] = scale(conc_norm[i])
        d[f'edge_size{suffix}'] = scale(weight_norm[i])

    return G


# -------------------------------
# RUN PIPELINE
# -------------------------------
G = build_contract_network(data, weight_mode="log_sum")

# Linear scaling
G_linear = add_visual_attributes(
    G.copy(),
    thickness_mode="linear",
    size_mode="linear",
    suffix="_linear"
)

# Log scaling
G_log = add_visual_attributes(
    G.copy(),
    thickness_mode="log",
    size_mode="log",
    suffix="_log"
)

In [11]:
def prepare_for_gephi(G):

    G_export = G.copy()

    # --- EDGE ATTRIBUTES ---
    for u, v, d in G_export.edges(data=True):

        for key, value in d.items():

            # convert lists to strings
            if isinstance(value, list):
                d[key] = "; ".join(map(str, value))

            # convert numpy types
            elif isinstance(value, np.generic):
                d[key] = value.item()

    # --- NODE ATTRIBUTES ---
    for n, d in G_export.nodes(data=True):

        for key, value in d.items():

            if isinstance(value, list):
                d[key] = "; ".join(map(str, value))

            elif isinstance(value, np.generic):
                d[key] = value.item()

    return G_export

G_gephi = prepare_for_gephi(G)

nx.write_gexf(G_gephi, "contracts_network.gexf")

## <font size=5>**3.1 City / Companies network**</font> <a class="anchor" id="3.1"></a>

which are the most dominant companies in the country over last 3 years?

[Back to TOC](#toc)

In [12]:
def build_city_contract_network(data) -> nx.MultiDiGraph:

    df = _ensure_dataframe(data)

    # --- CLEAN COLUMNS ---
    df = df.rename(columns={
        'precoContratual': 'price',
        'city': 'source',
        'adjudicatarios_clean': 'target'
    })

    # --- BASIC CLEANING ---
    df = df.dropna(subset=['source', 'target', 'price'])
    df = df[df['price'] > 0]

    # remove empty city names
    df = df[df['source'].astype(str).str.strip() != ""]

    # --- CONCORRENTES ---
    if 'nr_concorrentes' in df.columns:
        df['concorrentes'] = df['nr_concorrentes']
    else:
        df['concorrentes'] = pd.to_numeric(
            df.get('concorrentes', 0),
            errors='coerce'
        ).fillna(0)

    # --- BUILD MULTI GRAPH ---
    # each edge = one contract
    G = nx.MultiDiGraph()

    for row in df.itertuples(index=False):

        weight = compute_weight([row.price], mode="sum_price")

        G.add_edge(
            row.source,
            row.target,

            # unique edge key
            key=row.idcontrato,
            id=str(row.idcontrato),

            # edge attributes
            weight=weight,
            total_price=row.price,
            nr_concorrentes=row.concorrentes,
            contracts=1,

            idcontrato=row.idcontrato,
            tipoContrato=row.tipoContrato,
            tipoFimContrato=row.tipoFimContrato,
            CPV=row.CPV,

            precoBaseProcedimento=row.precoBaseProcedimento,
            precoContratual=row.price,
            PrecoTotalEfetivo=row.PrecoTotalEfetivo,

            dataDecisaoAdjudicacao=row.dataDecisaoAdjudicacao,
            dataCelebracaoContrato=row.dataCelebracaoContrato,
            dataPublicacao=row.dataPublicacao,
            dataFechoContrato=row.dataFechoContrato,

            contribuinte_adjudicante=row.contribuinte_adjudicante,
            contribuinte_adjudicatarios=row.contribuinte_adjudicatarios,

            adjudicante=row.adjudicante_clean,

            city=row.source,

            cpv_prefix=row.cpv_prefix,
            agg_cpv=row.agg_cpv
        )

    # --- NODE TYPES ---
    cities = set(df['source'])
    targets = set(df['target'])

    for node in G.nodes():

        if node in cities and node in targets:
            G.nodes[node]['node_type'] = 'both'

        elif node in cities:
            G.nodes[node]['node_type'] = 'city'

        else:
            G.nodes[node]['node_type'] = 'adjudicatario'

    return G

In [13]:
def add_visual_attributes_multigraph(
    G,
    thickness_mode="linear",
    size_mode="linear",
    scale_min=1,
    scale_max=5,
    suffix=""
):

    conc = np.array([
        d['nr_concorrentes']
        for _, _, _, d in G.edges(keys=True, data=True)
    ])

    weight = np.array([
        d['weight']
        for _, _, _, d in G.edges(keys=True, data=True)
    ])

    def normalize(x):

        if len(x) == 0:
            return x

        range_ = np.ptp(x)

        if range_ == 0:
            return np.zeros_like(x)

        return (x - np.min(x)) / (range_ + 1e-9)

    # --- TRANSFORM ---
    if thickness_mode == "log":
        conc = np.log1p(conc)

    if size_mode == "log":
        weight = np.log1p(weight)

    conc_norm = normalize(conc)
    weight_norm = normalize(weight)

    def scale(x):
        return scale_min + (scale_max - scale_min) * x

    # --- ASSIGN ---
    for i, (u, v, k, d) in enumerate(
        G.edges(keys=True, data=True)
    ):

        d[f'edge_thickness{suffix}'] = scale(conc_norm[i])
        d[f'edge_size{suffix}'] = scale(weight_norm[i])

    return G

G_city = build_city_contract_network(data)

G_city_linear = add_visual_attributes_multigraph(
    G_city.copy(),
    thickness_mode="linear",
    size_mode="linear",
    suffix="_linear"
)

G_city_log = add_visual_attributes_multigraph(
    G_city.copy(),
    thickness_mode="log",
    size_mode="log",
    suffix="_log"
)

In [14]:
from datetime import datetime

def prepare_for_gephi(G):

    G_export = G.copy()

    # --- EDGE ATTRIBUTES ---
    for u, v, k, d in G_export.edges(keys=True, data=True):

        for key, value in d.items():

            # lists -> string
            if isinstance(value, list):
                d[key] = "; ".join(map(str, value))

            # pandas timestamps / datetime
            elif isinstance(value, (pd.Timestamp, datetime)):
                d[key] = value.strftime("%Y-%m-%d")

            # numpy types
            elif isinstance(value, np.generic):
                d[key] = value.item()

    # --- NODE ATTRIBUTES ---
    for n, d in G_export.nodes(data=True):

        for key, value in d.items():

            if isinstance(value, list):
                d[key] = "; ".join(map(str, value))

            elif isinstance(value, (pd.Timestamp, datetime)):
                d[key] = value.strftime("%Y-%m-%d")

            elif isinstance(value, np.generic):
                d[key] = value.item()

    return G_export


G_gephi = prepare_for_gephi(G_city_log)

nx.write_gexf(
    G_gephi,
    "municipalities_network.gexf"
)

In [15]:
from collections import Counter

adj_in_degrees = [
    G_city.in_degree(n)
    for n, d in G_city.nodes(data=True)
    if d["node_type"] == "adjudicatario"
]

adj_degree_counts = Counter(adj_in_degrees)

total_adj_nodes = sum(adj_degree_counts.values())

adj_degree_distribution = (
    pd.DataFrame(
        adj_degree_counts.items(),
        columns=["in_degree", "num_nodes"]
    )
    .sort_values("in_degree")
    .reset_index(drop=True)
)

# add percentage column
adj_degree_distribution["percent"] = (
    adj_degree_distribution["num_nodes"] / total_adj_nodes * 100
).round(1)

adj_degree_distribution

,in_degree,num_nodes,percent
0,1,2581,52.8
1,2,902,18.4
2,3,434,8.9
3,4,236,4.8
4,5,182,3.7
5,6,113,2.3
6,7,87,1.8
7,8,54,1.1
8,9,38,0.8
9,10,35,0.7


In [16]:
import plotly.express as px

fig = px.bar(
    adj_degree_distribution,
    x="in_degree",
    y="num_nodes",
    title="In-Degree Distribution (Adjudicatarios)",
    labels={
        "in_degree": "In-degree",
        "num_nodes": "Number of nodes"
    }
)

fig.show()

In [17]:
import plotly.express as px

adj_in_degrees = [
    G_city.in_degree(n)
    for n, d in G_city.nodes(data=True)
    if d["node_type"] == "adjudicatario"
]

fig = px.box(
    y=adj_in_degrees,
    points="all",  # shows individual nodes as dots
    title="In-Degree Distribution (Adjudicatarios)",
    labels={"y": "In-degree"}
)

fig.show()

In [18]:
fig = px.box(
    y=adj_in_degrees,
    title="In-Degree Distribution (Log Perspective)",
    labels={"y": "In-degree"}
)

fig.update_yaxes(type="log")

fig.show()

In [19]:
adj_in_degree = {
    n: G_city.in_degree(n)
    for n, d in G_city.nodes(data=True)
    if d["node_type"] == "adjudicatario"
}
df_deg = pd.DataFrame(
    list(adj_in_degree.items()),
    columns=["company", "in_degree"]
)
Q1 = df_deg["in_degree"].quantile(0.25)
Q3 = df_deg["in_degree"].quantile(0.75)
IQR = Q3 - Q1

upper_bound = Q3 + 1.5 * IQR

outliers = df_deg[df_deg["in_degree"] > upper_bound]

top_outliers = (
    outliers.sort_values("in_degree", ascending=False)
    .head(10)
    .reset_index(drop=True)
)



fig = px.bar(
    top_outliers,
    x="in_degree",
    y="company",
    orientation="h",
    title="Top Dominant Adjudicatarios (Outliers by In-Degree)",
    labels={
        "in_degree": "Number of Cities Connected",
        "company": "Company"
    }
)

fig.update_layout(yaxis={"categoryorder": "total ascending"})
fig.show()

> Geographical constraint is a major limitation in public procurement competition

> There are a few companies that are able to overcome this limitation

____

## <font size=5>**3.1 CPV / Companies network**</font> <a class="anchor" id="3.1"></a>
  
[Back to TOC](#toc)

In [20]:
def build_cpv_contract_network(data) -> nx.MultiDiGraph:
    # Ensure dataframe (replaced your custom function with standard pandas)
    df = data.copy() if isinstance(data, pd.DataFrame) else pd.DataFrame(data)

    # Rename to set CPV as source and Company (adjudicatario) as target
    df = df.rename(columns={
        'precoContratual': 'price',
        'agg_cpv': 'source',
        'adjudicatarios_clean': 'target'
    })

    # Clean missing and invalid data
    df = df.dropna(subset=['source', 'target', 'price'])
    df = df[df['price'] > 0]

    df["source"] = df["source"].astype(str).str.strip()
    df["target"] = df["target"].astype(str).str.strip()
    df = df[(df["source"] != "") & (df["target"] != "")]

    G = nx.MultiDiGraph()

    # Build Edges
    for row in df.itertuples(index=False):
        G.add_edge(
            row.source,
            row.target,
            key=str(row.idcontrato),
            
            # Numeric/Weights
            weight=row.price,
            total_price=row.price,
            nr_concorrentes=getattr(row, "nr_concorrentes", 0),
            contracts=1,
            precoBaseProcedimento=row.precoBaseProcedimento,
            precoContratual=row.price,
            PrecoTotalEfetivo=row.PrecoTotalEfetivo,
            
            # Identifiers & Categories
            idcontrato=row.idcontrato,
            tipoContrato=row.tipoContrato,
            tipoFimContrato=row.tipoFimContrato,
            CPV=row.CPV,
            cpv_prefix=row.cpv_prefix,
            agg_cpv=row.source, # We renamed agg_cpv to 'source' above
            
            # Dates
            dataDecisaoAdjudicacao=row.dataDecisaoAdjudicacao,
            dataCelebracaoContrato=row.dataCelebracaoContrato,
            dataPublicacao=row.dataPublicacao,
            dataFechoContrato=row.dataFechoContrato,
            
            # Entities
            contribuinte_adjudicante=row.contribuinte_adjudicante,
            contribuinte_adjudicatarios=row.contribuinte_adjudicatarios,
            adjudicante=getattr(row, "adjudicante_clean", "")
        )

    # Assign Node Types (Optimized)
    # Extracting the set outside the loop drastically improves performance
    cpv_nodes = set(df["source"])
    
    for n in G.nodes():
        if n in cpv_nodes:
            G.nodes[n]["node_type"] = "cpv"
            G.nodes[n]["bipartite"] = 0 # Standard NetworkX bipartite flag
        else:
            G.nodes[n]["node_type"] = "adjudicatario"
            G.nodes[n]["bipartite"] = 1

    return G

In [21]:
def add_visual_attributes_multigraph(
    G,
    thickness_mode="linear",
    size_mode="linear",
    scale_min=1,
    scale_max=5,
    suffix=""
):
    conc = np.array([
        d.get('nr_concorrentes', 0)
        for _, _, _, d in G.edges(keys=True, data=True)
    ])

    weight = np.array([
        d.get('weight', 0)
        for _, _, _, d in G.edges(keys=True, data=True)
    ])

    def normalize(x):
        if len(x) == 0:
            return x
        range_ = np.ptp(x)
        if range_ == 0:
            return np.zeros_like(x)
        return (x - np.min(x)) / (range_ + 1e-9)

    # --- TRANSFORM ---
    if thickness_mode == "log":
        conc = np.log1p(conc)

    if size_mode == "log":
        weight = np.log1p(weight)

    conc_norm = normalize(conc)
    weight_norm = normalize(weight)

    def scale(x):
        return scale_min + (scale_max - scale_min) * x

    # --- ASSIGN ---
    for i, (u, v, k, d) in enumerate(G.edges(keys=True, data=True)):
        d[f'edge_thickness{suffix}'] = scale(conc_norm[i])
        d[f'edge_size{suffix}'] = scale(weight_norm[i])

    return G

In [22]:
def prepare_for_gephi(G):
    G_export = G.copy()

    # ---------------------------
    # EDGE ATTRIBUTES
    # ---------------------------
    for u, v, k, d in G_export.edges(keys=True, data=True):
        keys_to_convert = list(d.keys())

        for key in keys_to_convert:
            value = d[key]

            if isinstance(value, list) or isinstance(value, set):
                d[key] = "; ".join(map(str, value))
            elif isinstance(value, (pd.Timestamp, datetime)):
                d[key] = value.strftime("%Y-%m-%d")
            elif isinstance(value, np.generic):
                d[key] = value.item()
            elif pd.isna(value) or value is None: # Catches pandas NaN types safely
                d[key] = ""

        # Gephi requirement for MultiDiGraphs
        if "id" not in d:
            d["id"] = str(k)

    # ---------------------------
    # NODE ATTRIBUTES
    # ---------------------------
    for n, d in G_export.nodes(data=True):
        keys_to_convert = list(d.keys())

        for key in keys_to_convert:
            value = d[key]

            if isinstance(value, list) or isinstance(value, set):
                d[key] = "; ".join(map(str, value))
            elif isinstance(value, (pd.Timestamp, datetime)):
                d[key] = value.strftime("%Y-%m-%d")
            elif isinstance(value, np.generic):
                d[key] = value.item()
            elif pd.isna(value) or value is None:
                d[key] = ""

    return G_export

In [23]:
# 1. Build the network
G_cpv = build_cpv_contract_network(data)

# 2. Add visual scaling (Linear)
G_cpv_linear = add_visual_attributes_multigraph(
    G_cpv.copy(),
    thickness_mode="linear",
    size_mode="linear",
    suffix="_linear"
)

# 3. Add visual scaling (Logarithmic)
G_cpv_log = add_visual_attributes_multigraph(
    G_cpv.copy(),
    thickness_mode="log",
    size_mode="log",
    suffix="_log"
)

# 4. Prepare for Gephi Export
G_gephi = prepare_for_gephi(G_cpv)

# 5. Export to GEXF
nx.write_gexf(
    G_gephi,
    "cpv_network.gexf",
    version="1.2draft"
)

In [24]:
from collections import Counter
import pandas as pd

# --- compute in-degrees ---
adj_in_degrees = [
    G_cpv.in_degree(n)
    for n, d in G_cpv.nodes(data=True)
    if d["node_type"] == "adjudicatario"
]

# --- distribution ---
adj_degree_counts = Counter(adj_in_degrees)

adj_degree_distribution = (
    pd.DataFrame(
        adj_degree_counts.items(),
        columns=["in_degree", "num_nodes"]
    )
    .sort_values("in_degree")
    .reset_index(drop=True)
)

# --- correct total (IMPORTANT FIX) ---
total_nodes = len(adj_in_degrees)

# --- percentage ---
adj_degree_distribution["percent"] = (
    adj_degree_distribution["num_nodes"] / total_nodes * 100
).round(1)

adj_degree_distribution

,in_degree,num_nodes,percent
0,1,2738,51.3
1,2,957,17.9
2,3,484,9.1
3,4,270,5.1
4,5,203,3.8
5,6,139,2.6
6,7,98,1.8
7,8,61,1.1
8,9,49,0.9
9,10,38,0.7


In [25]:
import plotly.express as px

fig = px.bar(
    adj_degree_distribution,
    x="in_degree",
    y="num_nodes",
    title="In-Degree Distribution (Adjudicatarios)",
    labels={
        "in_degree": "In-degree",
        "num_nodes": "Number of nodes"
    }
)

fig.show()

In [26]:
import plotly.express as px

adj_in_degrees = [
    G_cpv.in_degree(n)
    for n, d in G_cpv.nodes(data=True)
    if d["node_type"] == "adjudicatario"
]

fig = px.box(
    y=adj_in_degrees,
    points="all",  # shows individual nodes as dots
    title="In-Degree Distribution (Adjudicatarios)",
    labels={"y": "In-degree"}
)

fig.show()

In [27]:
fig = px.box(
    y=adj_in_degrees,
    title="In-Degree Distribution (Log Perspective)",
    labels={"y": "In-degree"}
)

fig.update_yaxes(type="log")

fig.show()

In [28]:
from collections import Counter

specialization = {}

for company, d in G_cpv.nodes(data=True):

    if d["node_type"] != "adjudicatario":
        continue

    # all CPVs connected to company
    cpvs = [
        u
        for u, v, k in G_cpv.in_edges(company, keys=True)
    ]

    if len(cpvs) == 0:
        continue

    counts = Counter(cpvs)

    total = sum(counts.values())

    proportions = [
        c / total
        for c in counts.values()
    ]

    H = sum(p**2 for p in proportions)

    specialization[company] = H

df_spec = pd.DataFrame(
    specialization.items(),
    columns=["company", "specialization_index"]
)

fig = px.histogram(
    df_spec,
    x="specialization_index",
    nbins=30,
    title="Company Specialization Index"
)

fig.show()

pi = proportion of contracts in CPV i

<br>

| H value  | Meaning                         |
| -------- | ------------------------------- |
| ~1       | almost all contracts in one CPV |
| 0.5      | concentrated                    |
| 0.1      | diversified                     |
| very low | broad generalist                |

<br>

> Companies do not win contracts outside of their cpv area

In [28]:
adj_in_degree = {
    n: G_cpv.in_degree(n)
    for n, d in G_cpv.nodes(data=True)
    if d["node_type"] == "adjudicatario"
}
df_deg = pd.DataFrame(
    list(adj_in_degree.items()),
    columns=["company", "in_degree"]
)
Q1 = df_deg["in_degree"].quantile(0.25)
Q3 = df_deg["in_degree"].quantile(0.75)
IQR = Q3 - Q1

upper_bound = Q3 + 1.5 * IQR

outliers = df_deg[df_deg["in_degree"] > upper_bound]

top_outliers = (
    outliers.sort_values("in_degree", ascending=False)
    .head(10)
    .reset_index(drop=True)
)



fig = px.bar(
    top_outliers,
    x="in_degree",
    y="company",
    orientation="h",
    title="Top Dominant Adjudicatarios (Outliers by In-Degree)",
    labels={
        "in_degree": "Number of Cities Connected",
        "company": "Company"
    }
)

fig.update_layout(yaxis={"categoryorder": "total ascending"})
fig.show()

In [ ]:
import pandas as pd
import plotly.express as px
import ipywidgets as widgets
from IPython.display import display

# -----------------------------------
# AVAILABLE CPV AREAS
# -----------------------------------
cpv_options = sorted([
    n for n, d in G_cpv.nodes(data=True)
    if d["node_type"] == "cpv"
])

# -----------------------------------
# INTERACTIVE PLOT FUNCTION
# -----------------------------------
def plot_cpv_dominance(selected_cpv):

    # -----------------------------------
    # CONTRACT COUNT PER COMPANY
    # (inside selected CPV only)
    # -----------------------------------
    contracts_per_company = {}

    for _, company, k, d in G_cpv.edges(
        selected_cpv,
        keys=True,
        data=True
    ):

        contracts_per_company[company] = (
            contracts_per_company.get(company, 0) + 1
        )

    # -----------------------------------
    # DATAFRAME
    # -----------------------------------
    df_deg = pd.DataFrame(
        list(contracts_per_company.items()),
        columns=["company", "contracts"]
    )

    if df_deg.empty:
        print("No companies found for this CPV.")
        return

    # -----------------------------------
    # OUTLIER DETECTION
    # -----------------------------------
    Q1 = df_deg["contracts"].quantile(0.25)
    Q3 = df_deg["contracts"].quantile(0.75)
    IQR = Q3 - Q1

    upper_bound = Q3 + 1.5 * IQR

    outliers = df_deg[
        df_deg["contracts"] > upper_bound
    ]

    # fallback if no outliers
    if outliers.empty:
        outliers = df_deg

    top_outliers = (
        outliers
        .sort_values("contracts", ascending=False)
        .head(10)
    )

    # -----------------------------------
    # PLOT
    # -----------------------------------
    fig = px.bar(
        top_outliers.sort_values("contracts"),
        x="contracts",
        y="company",
        orientation="h",
        title=f"Top Dominant Companies — {selected_cpv}",
        labels={
            "contracts": "Number of Contracts",
            "company": "Company"
        }
    )

    fig.update_layout(
        height=600,
        yaxis={"categoryorder": "total ascending"}
    )

    fig.show()

# -----------------------------------
# DROPDOWN SELECTION
# -----------------------------------
dropdown = widgets.Dropdown(
    options=cpv_options,
    description="CPV:",
    layout=widgets.Layout(width="70%")
)

interactive_plot = widgets.interactive_output(
    plot_cpv_dominance,
    {"selected_cpv": dropdown}
)

display(dropdown, interactive_plot)

Dropdown(description='CPV:', layout=Layout(width='70%'), options=('Agricultura & recursos naturais', 'Ambiente…

Output()

In [29]:
from collections import Counter
import numpy as np
import pandas as pd
import plotly.express as px

# ----------------------------------------
# CITY DIVERSITY
# ----------------------------------------
# number of unique cities per company

city_diversity = {}

for n, d in G_city.nodes(data=True):

    if d["node_type"] != "adjudicatario":
        continue

    cities = set(G_city.predecessors(n))

    city_diversity[n] = len(cities)

# ----------------------------------------
# CPV ENTROPY
# ----------------------------------------

cpv_entropy = {}

for n, d in G_cpv.nodes(data=True):

    if d["node_type"] != "adjudicatario":
        continue

    cpvs = [
        u
        for u, v, k in G_cpv.in_edges(n, keys=True)
    ]

    if len(cpvs) == 0:
        continue

    counts = Counter(cpvs)

    total = sum(counts.values())

    proportions = [
        c / total
        for c in counts.values()
    ]

    H = -sum(p * np.log(p) for p in proportions)

    # normalized entropy
    if len(counts) > 1:
        H_norm = H / np.log(len(counts))
    else:
        H_norm = 0

    cpv_entropy[n] = H_norm

# ----------------------------------------
# MERGE METRICS
# ----------------------------------------

common_companies = (
    set(city_diversity.keys())
    & set(cpv_entropy.keys())
)

df_compare = pd.DataFrame({
    "company": list(common_companies),
    "city_diversity": [
        city_diversity[c]
        for c in common_companies
    ],
    "cpv_entropy": [
        cpv_entropy[c]
        for c in common_companies
    ]
})

# optional: log scale for city diversity
df_compare["log_city_diversity"] = np.log1p(
    df_compare["city_diversity"]
)

# ----------------------------------------
# SCATTER PLOT
# ----------------------------------------

fig = px.scatter(
    df_compare,
    x="city_diversity",
    y="cpv_entropy",
    hover_data=["company"],
    title="CPV Entropy vs City Diversity",
    labels={
        "city_diversity": "Number of Cities",
        "cpv_entropy": "Normalized CPV Entropy"
    }
)

fig.update_layout(
    height=700
)

fig.show()

---

In [40]:
weight_modes = ["sum_price", "avg_price", "log_sum", "log_mean"]

graphs = {
    mode: build_contract_network(data, weight_mode=mode)
    for mode in weight_modes
}

In [41]:
for mode, G in graphs.items():
    volume = {
        node: sum(d["total_price"] for _, _, d in G.edges(node, data=True))
        for node in G.nodes()
    }

    degree = dict(G.degree())

    corr = pd.Series(degree).corr(pd.Series(volume))

    print(f"{mode:10s} → degree-volume correlation: {corr:.3f}")

sum_price  → degree-volume correlation: 0.618
avg_price  → degree-volume correlation: 0.618
log_sum    → degree-volume correlation: 0.618
log_mean   → degree-volume correlation: 0.618


In [42]:
for mode, G in graphs.items():
    top = sorted(G.degree(), key=lambda x: x[1], reverse=True)[:3]

    print(f"\nMode: {mode}")
    for node, deg in top:
        print(f"  {node}: {deg}")


Mode: sum_price
  guarda nacional republicana: 62
  secretaria geral do ministerio da administracao interna: 26
  casa pia de lisboa i p: 18

Mode: avg_price
  guarda nacional republicana: 62
  secretaria geral do ministerio da administracao interna: 26
  casa pia de lisboa i p: 18

Mode: log_sum
  guarda nacional republicana: 62
  secretaria geral do ministerio da administracao interna: 26
  casa pia de lisboa i p: 18

Mode: log_mean
  guarda nacional republicana: 62
  secretaria geral do ministerio da administracao interna: 26
  casa pia de lisboa i p: 18


In [43]:
rankings = {}

for mode, G in graphs.items():

    weighted_degree = {
        node: sum(d["weight"] for _, _, d in G.edges(node, data=True))
        for node in G.nodes()
    }

    # convert to ranking (higher = better rank)
    ranked = pd.Series(weighted_degree).rank(ascending=False)

    rankings[mode] = ranked

sum_price → total contract value emphasis

avg_price → typical contract size

log_sum → interaction-frequency + heavy-tail compression

log_mean → normalized interaction intensity

In [44]:
rank_df = pd.DataFrame(rankings)
rank_df.head()

,sum_price,avg_price,log_sum,log_mean
acss administracao central do sistema de saude ip,95.0,93.0,61.0,57.0
claranet ii solutions,347.0,347.0,347.0,347.0
primavera business software solutions,347.0,347.0,347.0,347.0
timestamp sistemas de informacao,347.0,347.0,347.0,347.0
adene agencia para a energia,84.0,82.0,79.0,77.0


In [45]:
rank_df["rank_std"] = rank_df.std(axis=1)
rank_df["rank_mean"] = rank_df.mean(axis=1)

In [46]:
# Structurally important regardless of weight strategy

stable_nodes = rank_df.sort_values("rank_std").head(10)
stable_nodes

,sum_price,avg_price,log_sum,log_mean,rank_std,rank_mean
crc car rental company,347.0,347.0,347.0,347.0,0.0,277.6
claranet ii solutions,347.0,347.0,347.0,347.0,0.0,277.6
primavera business software solutions,347.0,347.0,347.0,347.0,0.0,277.6
timestamp sistemas de informacao,347.0,347.0,347.0,347.0,0.0,277.6
nova expressao planeamento de meios e publicidade,347.0,347.0,347.0,347.0,0.0,277.6
digiberia information technologies,347.0,347.0,347.0,347.0,0.0,277.6
paginas aos blocos,347.0,347.0,347.0,347.0,0.0,277.6
administracao central do sistema de saude i p,126.0,126.0,126.0,126.0,0.0,100.8
claranet portugal,347.0,347.0,347.0,347.0,0.0,277.6
onretrieval group,347.0,347.0,347.0,347.0,0.0,277.6


<font color ='red'>These nodes:
- have identical ranks across ALL weighting schemes
- are structurally invariant in your network

They are topological anchors, not sensitive to how we measure money or interaction.

We see identical rank values like 5594.5, this suggests many tied ranks. So they are stable and saturated centrality nodes. This means that ranking resolution is too coarse and many nodes are indistinguishable under this metric.

In [47]:
# Importende depends heavily on how we measure importance

unstable_nodes = rank_df.sort_values("rank_std", ascending=False).head(10)
unstable_nodes

,sum_price,avg_price,log_sum,log_mean,rank_std,rank_mean
area metropolitana de lisboa,53.0,117.0,10.0,122.0,53.792812,71.158562
instituto do emprego e formacao profissional ip,8.0,7.0,84.0,83.0,43.882419,45.176484
instituto nacional de saude doutor ricardo jorge i p,88.0,86.0,27.0,22.0,36.151302,51.830260
such | servico de utilizacao comum dos hospitais,62.0,106.0,23.0,78.0,34.654245,60.730849
municipio da lourinha,105.0,109.0,45.0,56.0,32.968419,69.593684
agencia nacional para a qualificacao e o ensino profissional i p,75.0,70.0,16.0,18.0,32.118271,42.223654
instituto superior de economia e gestao,66.0,63.0,11.0,8.0,31.801467,35.960293
cascais dinamica gestao de economia turismo e empreendorismo e m,30.0,29.0,85.0,84.0,31.759513,51.951903
adp valor servicos ambientais,34.0,32.0,86.0,85.0,30.324632,53.464926
faculdade de letras da universidade de lisboa,94.0,92.0,43.0,39.0,30.077677,59.615535


<font color='red'>These nodes change position dramatically depending on weighting scheme. Their importance depends on what we consider "importance". 

Pattern:
1. Mixed contract sizes
some large contracts
many small ones
2. irregular interaction structure
sporadic procurement behavior
3. heterogeneity in edges

sum_price -> rewards big contracts
avg_price -> smooths variability
log_sum -> rewards frequency
log_mean -> penalizes extreme dispersion

In [48]:
# # Visualize the graph
# # Layout (can take time for big graphs)
# pos = nx.spring_layout(G, k=0.15, iterations=20, seed=42)

# # Extract edge attributes
# weights = [G[u][v]['weight'] for u, v in G.edges()]
# widths = [G[u][v]['nr_concorrentes'] for u, v in G.edges()]

# # Avoid division by zero
# if len(widths) > 0 and max(widths) > 0:
#     widths = [w / max(widths) * 5 for w in widths]
# else:
#     widths = [1 for _ in widths]

# if len(weights) > 0 and max(weights) > 0:
#     weights_norm = [w / max(weights) for w in weights]
# else:
#     weights_norm = weights

# # Draw
# plt.figure(figsize=(12, 10))

# nx.draw_networkx_nodes(G, pos, node_size=50)

# nx.draw_networkx_edges(
#     G,
#     pos,
#     width=widths,                 # thickness = concorrentes
#     edge_color=weights_norm,      # color = log(price)
#     edge_cmap=plt.cm.Blues
# )

# plt.title("Network: Adjudicante → Adjudicatário")
# plt.axis('off')
# plt.show()

## <font size=5>**3.2 General Insights**</font> <a class="anchor" id="3.2"></a>
  
[Back to TOC](#toc)

In [49]:
# Number of nodes
print(f"Number of nodes: {G.number_of_nodes()}")
print(f"\t-> Number of adjudicantes: {sum(1 for _, attr in G.nodes(data=True) if attr['node_type'] in ['adjudicante', 'both'])}, i.e., {sum(1 for _, attr in G.nodes(data=True) if attr['node_type'] in ['adjudicante', 'both']) / G.number_of_nodes() * 100:.2f}%")
print(f"\t-> Number of adjudicatarios: {sum(1 for _, attr in G.nodes(data=True) if attr['node_type'] in ['adjudicatario', 'both'])}, i.e., {sum(1 for _, attr in G.nodes(data=True) if attr['node_type'] in ['adjudicatario', 'both']) / G.number_of_nodes() * 100:.2f}%")

# Number of edges
print(f"\nNumber of edges: {G.number_of_edges()}")

Number of nodes: 564
	-> Number of adjudicantes: 129, i.e., 22.87%
	-> Number of adjudicatarios: 436, i.e., 77.30%

Number of edges: 566


## <font size=5>**3.3 Rank All Entities By Degree (Number of Contracts)**</font> <a class="anchor" id="3.3"></a>
  
[Back to TOC](#toc)

In [50]:
def entity_type(node):
    node_type = G.nodes[node].get("node_type", "unknown")
    if node_type == "both":
        return "adjudicante and adjudicatario"
    return node_type

# Highest and lowest in-degree
highest_in = max(G.in_degree(), key=lambda x: x[1])
lowest_in = min(G.in_degree(), key=lambda x: x[1])

# Highest and lowest out-degree
highest_out = max(G.out_degree(), key=lambda x: x[1])
lowest_out = min(G.out_degree(), key=lambda x: x[1])

# Average in-degree and out-degree
in_degrees = [d for _, d in G.in_degree()]
out_degrees = [d for _, d in G.out_degree()]

avg_in = sum(in_degrees) / len(in_degrees)
avg_out = sum(out_degrees) / len(out_degrees)

print(f"Highest in-degree: {highest_in} -> {entity_type(highest_in[0])}")
print(f"Lowest out-degree: {lowest_out} -> {entity_type(lowest_out[0])}")

print(f"\nHighest out-degree: {highest_out} -> {entity_type(highest_out[0])}")
print(f"Lowest in-degree: {lowest_in} -> {entity_type(lowest_in[0])}")

print(f"\nAverage in-degree: {avg_in:.2f}")
print(f"Average out-degree: {avg_out:.2f}")

Highest in-degree: ('claranet ii solutions', 14) -> adjudicatario
Lowest out-degree: ('claranet ii solutions', 0) -> adjudicatario

Highest out-degree: ('guarda nacional republicana', 62) -> adjudicante
Lowest in-degree: ('acss administracao central do sistema de saude ip', 0) -> adjudicante

Average in-degree: 1.00
Average out-degree: 1.00


The average in-degree and average out-degree are the same because, in any directed graph, the sum of all in-degrees equals the sum of all out-degrees, and both are equal to the number of edges.

Therefore:
  $$avg_{in}  = |E| / |V|$$
  $$avg_{out} = |E| / |V|$$

so they must be identical.

In [51]:
# Top 5 Degree Enitities
top5 = sorted(G.degree(), key=lambda x: x[1], reverse=True)[:5]
print("Top 5 most connected entities:")
for entity, deg in top5:
    print(f"  {entity}: {deg}")

Top 5 most connected entities:
  guarda nacional republicana: 62
  secretaria geral do ministerio da administracao interna: 26
  casa pia de lisboa i p: 18
  servicos municipalizados de agua e saneamento de sintra: 18
  instituto nacional de medicina legal e ciencias forenses i p: 17


In [52]:
# Degree-1 entities
deg1 = sorted(
    [(node, degree) for node, degree in G.degree() if degree == 1],
    key=lambda x: x[0]
)

print(f"Degree-1 entities: {len(deg1)}")
for i, (entity, degree) in enumerate(deg1[:20], start=1):
    node_type = G.nodes[entity].get("node_type", "unknown")
    if node_type == "both":
        node_type = "adjudicante and adjudicatario"
    print(f"{i:>3}. {entity} (degree={degree}, type={node_type})")

if len(deg1) > 20:
    print(f"... and {len(deg1) - 20} more")

Degree-1 entities: 407
  1. 1 meo servicos de comunicacoes e multimedia (degree=1, type=adjudicatario)
  2. 2wayview (degree=1, type=adjudicatario)
  3. a barreira (degree=1, type=adjudicatario)
  4. a do carmo importacao exportacao e comercio (degree=1, type=adjudicatario)
  5. a gouv reparacao e manutencao de veiculos (degree=1, type=adjudicatario)
  6. a salgado distribuicao (degree=1, type=adjudicatario)
  7. abrancongelados produtos alimentares (degree=1, type=adjudicatario)
  8. adelaide ferreira (degree=1, type=adjudicatario)
  9. adelino pedro (degree=1, type=adjudicatario)
 10. administracao central do sistema de saude i p (degree=1, type=adjudicante)
 11. adp valor servicos ambientais (degree=1, type=adjudicante)
 12. ageas portugal companhia de seguros (degree=1, type=adjudicatario)
 13. agencia para o investimento e comercio externo de portugal e p e (degree=1, type=adjudicante)
 14. agripublic (degree=1, type=adjudicatario)
 15. agrupamento de escolas jose afonso loures (d

In [53]:
# All contract pairs with amounts
print("Contracts with their amounts:")
for u, v, data in G.edges(data=True):
    print(f"  {u} -- {v} : €{data['weight']:,}")

Contracts with their amounts:
  acss administracao central do sistema de saude ip -- claranet ii solutions : €8.433811582477187
  acss administracao central do sistema de saude ip -- primavera business software solutions : €8.990615968707628
  acss administracao central do sistema de saude ip -- timestamp sistemas de informacao : €11.34954392823967
  adene agencia para a energia -- digiberia information technologies : €11.923709734555993
  adene agencia para a energia -- paginas aos blocos : €8.160932447399158
  administracao central do sistema de saude i p -- claranet portugal : €6.763908002983978
  adp valor servicos ambientais -- hccm consulting : €13.283143542126385
  agencia nacional para a qualificacao e o ensino profissional i p -- chief security officers : €9.193273585525748
  agencia nacional para a qualificacao e o ensino profissional i p -- datagate desenvolvimento solucoes informaticas : €7.461640392208575
  agencia nacional para a qualificacao e o ensino profissional i p -

## <font size=5>**3.4 Total Contracts Volume (Price) per Entity**</font> <a class="anchor" id="3.4"></a>
  
[Back to TOC](#toc)

In [54]:
volume = {node: 0 for node in G.nodes()}

# Step 2
for u, v, data in G.edges(data=True):
    volume[u] += data["weight"]
    volume[v] += data["weight"]

# Step 3
sorted_volume = sorted(volume.items(), key=lambda x: x[1], reverse=True)
print("Entity volume ranking:")
for entity, vol in sorted_volume:
    print(f"  {entity}: €{vol:,.0f}")

Entity volume ranking:
  guarda nacional republicana: €606
  secretaria geral do ministerio da administracao interna: €278
  servicos municipalizados de agua e saneamento de sintra: €203
  gebalis gestao do arrendamento da habitacao municipal de lisboa e m: €174
  casa pia de lisboa i p: €173
  centro de formacao profissional das pescas e do mar for mar: €171
  instituto nacional de medicina legal e ciencias forenses i p: €149
  claranet ii solutions: €145
  instituto superior de economia e gestao: €142
  santa casa da misericordia de lisboa: €136
  agencia para a modernizacao administrativa i p: €131
  instituto de acao social das forcas armadas i p: €119
  servicos intermunicipalizados de aguas e residuos dos municipios de loures e odivelas: €119
  municipio de mafra: €112
  ministerio da defesa nacional marinha: €111
  municipio de oeiras: €99
  iseg instituto superior de economia e gestao: €99
  instituto nacional de estatistica i p: €92
  agencia nacional para a qualificacao e o e

## <font size=5>**3.5 Degree VS Volume**</font> <a class="anchor" id="3.5"></a>
  
[Back to TOC](#toc)

- Degree $\rightarrow$ number of contracts associated with each entity (network degree = in_degree + out_degree)
- Volume $\rightarrow$ total contracted amount associated with each entity (sum of incident edge weights)

**Purpose**: assess which entities are most valuable and how activity (degree) relates to economic impact (volume).

Planned steps:
- compute degree and volume per node
- plot degree vs. volume (use log–log scatter), annotate top entities
- report Pearson and Spearman correlations
- fit a linear model on log-transformed values and flag outliers (high volume/low degree and high degree/low volume)

In [55]:
# Degree and volume per node
metrics = pd.DataFrame({
    "degree": pd.Series(dict(G.degree())),
    "volume": pd.Series(volume)
}).fillna(0)

In [56]:
# Keep only positive values for log-log analysis
pos = metrics[(metrics["degree"] > 0) & (metrics["volume"] > 0)].copy()
pos["log_degree"] = np.log10(pos["degree"])
pos["log_volume"] = np.log10(pos["volume"])

# Correlations on log-transformed values
pearson = pos["log_degree"].corr(pos["log_volume"], method="pearson")
spearman = pos["log_degree"].corr(pos["log_volume"], method="spearman")

print(f"Nodes used in log-log analysis: {len(pos)}")
print(f"Pearson correlation (log10):  {pearson:.4f}")
print(f"Spearman correlation (log10): {spearman:.4f}")

Nodes used in log-log analysis: 564
Pearson correlation (log10):  0.9660
Spearman correlation (log10): 0.7834


In [57]:
# Linear model in log-log space
slope, intercept = np.polyfit(pos["log_degree"], pos["log_volume"], 1)
pos["pred_log_volume"] = intercept + slope * pos["log_degree"]
pos["residual"] = pos["log_volume"] - pos["pred_log_volume"]

print(f"\nlog10(volume) = {intercept:.4f} + {slope:.4f} * log10(degree)")


log10(volume) = 1.0157 + 1.0094 * log10(degree)


In [58]:
# Flag outliers using the box-plot rule (IQR)
degree_dict = dict(G.degree())

q1 = pd.Series(volume).quantile(0.25)
q3 = pd.Series(volume).quantile(0.75)
iqr = q3 - q1
volume_threshold = q3 + 1.5 * iqr

print(f"Volume threshold from box plot rule: €{volume_threshold:,.0f}")
print("Flagged entities (degree ≥ 3 AND volume above upper fence):")

flagged = []
for entity in G.nodes():
    if degree_dict[entity] >= 3 and volume[entity] > volume_threshold:
        flagged.append((entity, degree_dict[entity], volume[entity]))

flagged.sort(key=lambda x: x[2], reverse=True)
for entity, deg, vol in flagged:
    print(f"  {entity}  degree={deg}  volume=€{vol:,.0f}")

print(f"\nTotal flagged: {len(flagged)}")

Volume threshold from box plot rule: €35
Flagged entities (degree ≥ 3 AND volume above upper fence):
  guarda nacional republicana  degree=62  volume=€606
  secretaria geral do ministerio da administracao interna  degree=26  volume=€278
  servicos municipalizados de agua e saneamento de sintra  degree=18  volume=€203
  gebalis gestao do arrendamento da habitacao municipal de lisboa e m  degree=16  volume=€174
  casa pia de lisboa i p  degree=18  volume=€173
  centro de formacao profissional das pescas e do mar for mar  degree=16  volume=€171
  instituto nacional de medicina legal e ciencias forenses i p  degree=17  volume=€149
  claranet ii solutions  degree=14  volume=€145
  instituto superior de economia e gestao  degree=16  volume=€142
  santa casa da misericordia de lisboa  degree=15  volume=€136
  agencia para a modernizacao administrativa i p  degree=13  volume=€131
  instituto de acao social das forcas armadas i p  degree=11  volume=€119
  servicos intermunicipalizados de aguas 

In [59]:
# Plotting

# prepare fit line
x_line = np.logspace(np.log10(pos["degree"].min()), np.log10(pos["degree"].max()), 200)
y_line = 10 ** (intercept + slope * np.log10(x_line))

# main traces
scatter = go.Scatter(
    x=pos["degree"],
    y=pos["volume"],
    mode="markers",
    marker=dict(size=6, color="steelblue", opacity=0.5),
    name="nodes",
    hovertemplate="%{text}<br>Degree: %{x}<br>Volume: €%{y:,.0f}",
    text=pos.index
)

fit_line = go.Scatter(
    x=x_line,
    y=y_line,
    mode="lines",
    line=dict(color="crimson", width=2),
    name="log-log fit"
)

# annotations for top entities
annotations = []
top_entities = [entity for entity, _ in top5]

for node in top_entities:
    x = metrics.loc[node, "degree"]
    y = metrics.loc[node, "volume"]
    annotations.append(
        dict(
            x=x,
            y=y,
            text=node,
            showarrow=True,
            arrowhead=2,
            ax=10,
            ay=-10,
            font=dict(size=10),
        )
    )

fig = go.Figure(data=[scatter, fit_line])
fig.update_layout(
    title="Degree VS Volume",
    xaxis=dict(title="Degree", type="log"),
    yaxis=dict(title="Volume (€)", type="log"),
    annotations=annotations,
    legend=dict(yanchor="top", y=0.99, xanchor="left", x=0.01),
    margin=dict(l=60, r=20, t=60, b=60)
)

fig.update_layout(
    title="Degree vs Volume",
    
    xaxis=dict(
        title="Degree",
        type="log",
        dtick=1,              # only 10^n ticks → 1, 10, 100, 1000
        tickformat=".0f"      # show full numbers instead of scientific notation
    ),
    
    yaxis=dict(
        title="Volume (€)",
        type="log",
        dtick=1,
        # tickformat=".0f"
    ),

    annotations=annotations,
    legend=dict(yanchor="top", y=0.99, xanchor="left", x=0.01),
    margin=dict(l=60, r=20, t=60, b=60)
)

fig.update_xaxes(showgrid=True, gridwidth=1)
fig.update_yaxes(showgrid=True, gridwidth=1)

fig.show()

In [60]:
# TODO: add attribute adjudicante/adjudicatario to hoover in plot and maybe change the color of the point based on it 

**NETWORK MUNICIPALITY**

In [61]:
import pandas as pd
import numpy as np
import networkx as nx


# =========================================================
# 1. INPUT HANDLING (DICT / LIST / DF SAFE)
# =========================================================
def ensure_dataframe(data):

    if isinstance(data, pd.DataFrame):
        return data.copy()

    if isinstance(data, list):
        return pd.DataFrame(data)

    if isinstance(data, dict):
        # dict of dataframes
        for v in data.values():
            if isinstance(v, pd.DataFrame):
                return v.copy()

        # dict of records
        return pd.DataFrame(data)

    raise TypeError(f"Unsupported input type: {type(data)}")


# =========================================================
# 2. WEIGHT FUNCTION
# =========================================================
def compute_weight(x, mode="log_sum"):
    x = np.array(x)

    if mode == "log_sum":
        return np.log1p(x).sum()
    elif mode == "sum":
        return x.sum()
    elif mode == "mean":
        return x.mean()
    else:
        raise ValueError("Unknown weight mode")


def standardize_contract_data(df):

    df = df.copy()

    # =====================================================
    # 1. SOURCE (company)
    # =====================================================
    if 'source_company' in df.columns:
        pass

    elif 'adjudicante' in df.columns:
        df['source_company'] = df['adjudicante']

    elif 'source' in df.columns:
        df['source_company'] = df['source']

    elif 'adjudicatarios' in df.columns:
        # fallback case where structure is inverted/mixed
        df['source_company'] = df['adjudicante'] if 'adjudicante' in df.columns else np.nan

    else:
        raise ValueError(f"Cannot find source column. Columns: {df.columns.tolist()}")

    # =====================================================
    # 2. TARGET (company)
    # =====================================================
    if 'target_company' in df.columns:
        pass

    elif 'adjudicatarios' in df.columns:
        df['target_company'] = df['adjudicatarios']

    elif 'target' in df.columns:
        df['target_company'] = df['target']

    else:
        raise ValueError(f"Cannot find target column. Columns: {df.columns.tolist()}")

    # =====================================================
    # 3. PRICE
    # =====================================================
    if 'price' in df.columns:
        pass

    elif 'precoContratual' in df.columns:
        df['price'] = df['precoContratual']

    else:
        raise ValueError(f"Cannot find price column. Columns: {df.columns.tolist()}")

    # =====================================================
    # 4. CLEAN
    # =====================================================
    required = ['source_company', 'target_company', 'price']

    df = df.dropna(subset=required)
    df = df[df['price'] > 0]

    return df





# =========================================================
# 5. MUNICIPALITY NETWORK
# =========================================================
def build_municipality_network(data, weight_mode="log_sum"):

    df = ensure_dataframe(data)
    df = standardize_contract_data(df)

    if 'city' not in df.columns:
        raise ValueError("Missing 'city' column")

    df['source_municipality'] = df['city']
    df['target_company'] = df['target_company']

    edge_df = (
        df.groupby(['source_municipality', 'target_company'], as_index=False)
        .agg(
            total_price=('price', 'sum'),
            contracts=('price', 'count'),
            avg_price=('price', 'mean'),
            price_series=('price', list)
        )
    )

    edge_df['weight'] = edge_df['price_series'].apply(
        lambda x: compute_weight(x, mode=weight_mode)
    )

    G = nx.DiGraph()

    for row in edge_df.itertuples(index=False):
        G.add_edge(
            row.source_municipality,
            row.target_company,
            weight=row.weight,
            total_price=row.total_price,
            contracts=row.contracts,
            avg_price=row.avg_price
        )

    return G


# =========================================================
# 6. VISUAL ATTRIBUTES (OPTIONAL)
# =========================================================
def add_visual_attributes(
    G,
    thickness_mode="linear",
    size_mode="linear",
    scale_min=1,
    scale_max=5,
    suffix=""
):

    conc = np.array([d.get('nr_concorrentes', 0) for _, _, d in G.edges(data=True)])
    weight = np.array([d['weight'] for _, _, d in G.edges(data=True)])

    def normalize(x):
        if len(x) == 0:
            return x

        r = np.ptp(x)
        if r == 0:
            return np.zeros_like(x)

        return (x - np.min(x)) / (r + 1e-9)

    if thickness_mode == "log":
        conc = np.log1p(conc)

    if size_mode == "log":
        weight = np.log1p(weight)

    conc_norm = normalize(conc)
    weight_norm = normalize(weight)

    def scale(x):
        return scale_min + (scale_max - scale_min) * x

    for i, (_, _, d) in enumerate(G.edges(data=True)):
        d[f'edge_thickness{suffix}'] = scale(conc_norm[i])
        d[f'edge_size{suffix}'] = scale(weight_norm[i])

    return G


# =========================================================
# 7. PIPELINE USAGE
# =========================================================

df = data  # raw input (dict/list/DataFrame supported)

# MUNICIPALITY NETWORK
G_municipality = build_municipality_network(df)
G_municipality = add_visual_attributes(G_municipality)

ValueError: Cannot find source column. Columns: ['weight', 'total_price', 'nr_concorrentes', 'contracts', 'weight_mode', 'idcontrato', 'tipoContrato', 'tipoFimContrato', 'CPV', 'precoBaseProcedimento', 'precoContratual', 'PrecoTotalEfetivo', 'dataDecisaoAdjudicacao', 'dataCelebracaoContrato', 'dataPublicacao', 'dataFechoContrato', 'nr_concorrentes_list', 'contribuinte_adjudicante', 'contribuinte_adjudicatarios', 'city', 'cpv_prefix', 'agg_cpv']

In [40]:
df = _ensure_dataframe(data)
print(df.columns)

Index(['weight', 'total_price', 'nr_concorrentes', 'contracts', 'weight_mode',
       'idcontrato', 'tipoContrato', 'tipoFimContrato', 'CPV',
       'precoBaseProcedimento', 'precoContratual', 'PrecoTotalEfetivo',
       'dataDecisaoAdjudicacao', 'dataCelebracaoContrato', 'dataPublicacao',
       'dataFechoContrato', 'nr_concorrentes_list', 'contribuinte_adjudicante',
       'contribuinte_adjudicatarios', 'city', 'cpv_prefix', 'agg_cpv'],
      dtype='str')


## <font size=5>**3.6 Exporting It**</font> <a class="anchor" id="3.6"></a>
  
[Back to TOC](#toc)

In [31]:
output_path = '../graphs/gephi_graph01.gexf'
os.makedirs(os.path.dirname(output_path), exist_ok=True)
nx.write_gexf(G, output_path)
print(f'Graph exported to {output_path}')

TypeError: cannot unpack non-iterable int object